In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import  warnings
from sklearn.preprocessing import (StandardScaler, RobustScaler,
                                   MinMaxScaler, LabelEncoder, OrdinalEncoder)

warnings.filterwarnings("ignore")

In [ ]:
# Reading the file:
df = pd.read_csv('/content/drive/MyDrive/AI/employee_attrition.csv')
df.head()

,age,gender,department,education,years_experience,salary,performance_score,satisfaction_score,work_life_balance,monthly_hours,num_projects,distance_from_home,attrition
0,50,Male,HR,High School,27,101732.0,3,3.0,Excellent,146.0,4,20.0,0
1,36,Female,HR,Master,11,67955.0,4,3.0,Good,116.0,1,11.0,0
2,29,Male,Sales,Master,6,54796.0,3,3.0,Fair,206.0,4,15.0,0
3,42,Female,Operations,Bachelor,19,NaN,5,4.0,Good,138.0,3,5.0,0
4,40,Female,Finance,PhD,18,84908.0,5,3.0,Excellent,190.0,7,377.0,0


In [ ]:
print(df.isna().sum().sort_values(ascending=False))

satisfaction_score    70
distance_from_home    60
salary                50
monthly_hours         40
age                    0
years_experience       0
education              0
department             0
gender                 0
work_life_balance      0
performance_score      0
num_projects           0
attrition              0
dtype: int64


In [ ]:
df_clean = df.copy()

print("Before imputation:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Salary — skewed due to outliers → use MEDIAN
df_clean["salary"].fillna(df_clean["salary"].median(), inplace=True)

# satisfaction_score — discrete scale → use MODE
df_clean["satisfaction_score"].fillna(df_clean["satisfaction_score"].mode()[0], inplace=True)

# monthly_hours — approx normal → use MEAN
df_clean["monthly_hours"].fillna(df_clean["monthly_hours"].mean(), inplace=True)

# distance_from_home — skewed → use MEDIAN
df_clean["distance_from_home"].fillna(df_clean["distance_from_home"].median(), inplace=True)

print("\nAfter imputation:")
print(df_clean.isnull().sum())
print("\n✅ No missing values remaining!")

Before imputation:
salary                50
satisfaction_score    70
monthly_hours         40
distance_from_home    60
dtype: int64

After imputation:
age                   0
gender                0
department            0
education             0
years_experience      0
salary                0
performance_score     0
satisfaction_score    0
work_life_balance     0
monthly_hours         0
num_projects          0
distance_from_home    0
attrition             0
dtype: int64

✅ No missing values remaining!


In [ ]:
df_enc = df_clean.copy()

# Gender: Male=1, Female=0
le = LabelEncoder()
df_enc["gender_encoded"] = le.fit_transform(df_enc["gender"])
print("Gender encoding mapping:")
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} → {code}")
print(df_enc[["gender","gender_encoded"]].head(4))

Gender encoding mapping:
  Female → 0
  Male → 1
   gender  gender_encoded
0    Male               1
1  Female               0
2    Male               1
3  Female               0


In [ ]:
# Education: High School < Bachelor < Master < PhD
edu_order = [["High School", "Bachelor", "Master", "PhD"]]
oe = OrdinalEncoder(categories=edu_order)
df_enc["education_encoded"] = oe.fit_transform(df_enc[["education"]]).astype(int)

print("Education ordinal encoding:")
for i, cat in enumerate(edu_order[0]):
    print(f"  {cat} → {i}")
print(df_enc[["education","education_encoded"]].head(5))

Education ordinal encoding:
  High School → 0
  Bachelor → 1
  Master → 2
  PhD → 3
     education  education_encoded
0  High School                  0
1       Master                  2
2       Master                  2
3     Bachelor                  1
4          PhD                  3


In [ ]:
# Select only numeric ML-ready features
scale_cols = ["age", "years_experience", "salary", "monthly_hours",
              "distance_from_home", "num_projects",
              "performance_score", "satisfaction_score"]

X = df_enc[scale_cols].copy()
y = df_enc["attrition"].copy()

print("Feature matrix X shape:", X.shape)
print("Target vector y shape: ", y.shape)
print()
print(X.describe().T[["mean","std","min","max"]].round(1))

Feature matrix X shape: (1000, 8)
Target vector y shape:  (1000,)

                       mean      std      min       max
age                    40.9     11.2     22.0      59.0
years_experience       19.1     11.3      0.0      38.0
salary              86607.2  46628.6  14365.0  490111.0
monthly_hours         177.5     36.6     82.0     486.0
distance_from_home     14.8     32.2      1.0     377.0
num_projects            4.0      2.0      1.0       7.0
performance_score       3.4      1.0      1.0       5.0
satisfaction_score      3.4      1.1      1.0       5.0


In [ ]:
X.shape

(1000, 8)

In [ ]:
y.value_counts()

,count
attrition,
0,880
1,120


In [ ]:
# Splitting the Data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
                                                    )

In [ ]:
# Fit scaler on TRAIN only, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)    # fit + transform on train
X_test_scaled  = scaler.transform(X_test)          # transform only on test (no fitting!)

print("Scaler fit on train only — no data leakage!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled  shape: {X_test_scaled.shape}")

Scaler fit on train only — no data leakage!
X_train_scaled shape: (800, 8)
X_test_scaled  shape: (200, 8)


In [ ]:
# Build your Network

model = tf.keras.Sequential()

# Adding the Neurons:

model.add(tf.keras.layers.Input(shape=(8,)))
model.add(tf.keras.layers.Dense(100, activation="relu"))
model.add(tf.keras.layers.Dense(100, activation="relu"))
model.add(tf.keras.layers.Dense(1, activation="sigmoid"))

In [ ]:
# Summary:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │           900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,101 (43.36 KB)

 Trainable params: 11,101 (43.36 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
# Training the model
history = model.fit(X_train_scaled, y_train,
                    epochs=50,
                    batch_size=64,
                    validation_split=0.2,
                    verbose=1)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.8859 - loss: 0.3250 - val_accuracy: 0.8625 - val_loss: 0.4238
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8859 - loss: 0.3249 - val_accuracy: 0.8625 - val_loss: 0.4239
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8859 - loss: 0.3247 - val_accuracy: 0.8625 - val_loss: 0.4239
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8859 - loss: 0.3246 - val_accuracy: 0.8625 - val_loss: 0.4240
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8859 - loss: 0.3246 - val_accuracy: 0.8625 - val_loss: 0.4240
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8859 - loss: 0.3246 - val_accuracy: 0.8625 - val_loss: 0.4241
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8859 - loss: 0.3245 - val_accuracy: 0.8625 - val_loss: 0.4241
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8859 - loss: 0.3244 - val_accuracy: 0.8625 - val_los

In [ ]:
# Evaluation
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Accuracy: {test_accuracy *100:.2f}")

Accuracy: 88.50
